In [ ]:
import os
import time
from pathlib import Path

import numpy as np

# Check onnxruntime availability
try:
    import onnxruntime

    print(f"onnxruntime {onnxruntime.__version__} available")
except ImportError:
    print("onnxruntime not available — installing...")
    import subprocess

    subprocess.run(["pip", "install", "onnxruntime"], check=True)
    import onnxruntime

    print(f"onnxruntime {onnxruntime.__version__} installed")

# Detect ONNX model path
ONNX_MODEL_PATH = None
for p in Path("/kaggle/input").glob("**/perch*.onnx"):
    ONNX_MODEL_PATH = str(p)
    break
if ONNX_MODEL_PATH is None:
    for p in Path("/kaggle/input").glob("**/*.onnx"):
        ONNX_MODEL_PATH = str(p)
        break

# Detect TF SavedModel path
TF_MODEL_PATH = None
for p in Path("/kaggle/input").glob("**/perch_v2_cpu"):
    TF_MODEL_PATH = str(p)
    break
if TF_MODEL_PATH is None:
    for p in Path("/kaggle/input").glob("**/saved_model.pb"):
        TF_MODEL_PATH = str(p.parent)
        break

# Find soundscape test files
AUDIO_DIR = None
for p in [
    Path("/kaggle/input/birdclef-2026/test_soundscapes"),
    Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes"),
]:
    if p.exists():
        AUDIO_DIR = p
        break

print(f"ONNX: {ONNX_MODEL_PATH}")
print(f"TF:   {TF_MODEL_PATH}")
print(f"Audio: {AUDIO_DIR}")

In [ ]:
import librosa

N_BENCH = 20  # number of files to benchmark
SAMPLE_RATE = 32000
WINDOW_SEC = 5

files = sorted(list(AUDIO_DIR.glob("*.ogg")))[:N_BENCH]
print(f"Benchmark files: {len(files)}")

# Pre-load all audio to separate disk I/O from inference timing
audio_list = []
for f in files:
    y, sr = librosa.load(str(f), sr=SAMPLE_RATE, mono=True)
    # Pad/trim to exactly 60s
    target_len = 60 * SAMPLE_RATE
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    audio_list.append(y)

print(f"Loaded {len(audio_list)} files, shape example: {audio_list[0].shape}")

In [ ]:
# TF SavedModel inference
import tensorflow as tf

tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)

bird_model = tf.saved_model.load(TF_MODEL_PATH)
infer_fn = bird_model.signatures["serving_default"]


def run_tf_inference(audio_list):
    all_embeddings = []
    all_logits = []
    t0 = time.perf_counter()
    for audio in audio_list:
        windows = audio.reshape(12, WINDOW_SEC * SAMPLE_RATE)
        for window in windows:
            inp = tf.constant(window[np.newaxis], dtype=tf.float32)
            out = infer_fn(inp)
            emb = None
            logits = None
            # Get embeddings and logits - find keys by shape
            for k, v in out.items():
                if v.shape[-1] == 1536:
                    emb = v.numpy()
                elif v.shape[-1] > 1000:
                    logits = v.numpy()
            if emb is not None:
                all_embeddings.append(emb)
            if logits is not None:
                all_logits.append(logits)
    elapsed = time.perf_counter() - t0
    return elapsed, all_embeddings, all_logits


# Print output keys for debugging
sample_inp = tf.constant(
    audio_list[0][: WINDOW_SEC * SAMPLE_RATE][np.newaxis], dtype=tf.float32
)
sample_out = infer_fn(sample_inp)
print("TF output keys and shapes:")
for k, v in sample_out.items():
    print(f"  {k}: {v.shape}")

# Warmup
_ = run_tf_inference(audio_list[:1])
print("TF warmup done")

# Benchmark
tf_elapsed, tf_embeddings, tf_logits = run_tf_inference(audio_list)
tf_per_file = tf_elapsed / len(audio_list)
tf_extrap = tf_per_file * 739  # extrapolate to full test set

print(f"TF SavedModel: {tf_elapsed:.1f}s for {N_BENCH} files")
print(f"  Per file: {tf_per_file:.2f}s")
print(f"  Extrapolated to 739 files: {tf_extrap / 60:.1f} min")

In [ ]:
import onnxruntime as ort

# ONNX Runtime session options
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 2
sess_options.inter_op_num_threads = 2
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

ort_session = ort.InferenceSession(ONNX_MODEL_PATH, sess_options)

# Get input/output names
input_name = ort_session.get_inputs()[0].name
print(f"ONNX inputs:  {[(i.name, i.shape, i.type) for i in ort_session.get_inputs()]}")
print(f"ONNX outputs: {[(o.name, o.shape, o.type) for o in ort_session.get_outputs()]}")


def run_onnx_inference(audio_list):
    all_embeddings = []
    all_logits = []
    t0 = time.perf_counter()
    for audio in audio_list:
        windows = audio.reshape(12, WINDOW_SEC * SAMPLE_RATE)
        for window in windows:
            inp = window[np.newaxis].astype(np.float32)
            outputs = ort_session.run(None, {input_name: inp})
            # outputs is a list - find embedding (1536) and logit outputs by shape
            for out in outputs:
                if out.shape[-1] == 1536:
                    all_embeddings.append(out)
                elif out.shape[-1] > 1000:
                    all_logits.append(out)
    elapsed = time.perf_counter() - t0
    return elapsed, all_embeddings, all_logits


# Warmup
_ = run_onnx_inference(audio_list[:1])
print("ONNX warmup done")

# Benchmark
onnx_elapsed, onnx_embeddings, onnx_logits = run_onnx_inference(audio_list)
onnx_per_file = onnx_elapsed / len(audio_list)
onnx_extrap = onnx_per_file * 739

print(f"ONNX Runtime: {onnx_elapsed:.1f}s for {N_BENCH} files")
print(f"  Per file: {onnx_per_file:.2f}s")
print(f"  Extrapolated to 739 files: {onnx_extrap / 60:.1f} min")

In [ ]:
speedup = tf_elapsed / onnx_elapsed

print("=" * 50)
print(f"SPEEDUP: {speedup:.2f}x")
print(f"TF:   {tf_per_file:.2f}s/file → {tf_extrap / 60:.0f} min for 739 files")
print(f"ONNX: {onnx_per_file:.2f}s/file → {onnx_extrap / 60:.0f} min for 739 files")
print()

# Verify output similarity
if tf_embeddings and onnx_embeddings:
    tf_emb = np.vstack(tf_embeddings)
    onnx_emb = np.vstack(onnx_embeddings)
    # Align length in case of mismatch
    n = min(len(tf_emb), len(onnx_emb))
    tf_emb = tf_emb[:n]
    onnx_emb = onnx_emb[:n]
    # cosine similarity
    tf_norm = tf_emb / np.linalg.norm(tf_emb, axis=-1, keepdims=True)
    onnx_norm = onnx_emb / np.linalg.norm(onnx_emb, axis=-1, keepdims=True)
    cos_sim = (tf_norm * onnx_norm).sum(-1).mean()
    max_abs_diff = np.abs(tf_emb - onnx_emb).max()
    print(f"Embedding cosine similarity: {cos_sim:.6f} (want >0.999)")
    print(f"Max absolute diff:           {max_abs_diff:.6f}")
    print()

if speedup >= 3:
    remaining = 90 - onnx_extrap / 60
    print(
        f"✅ ONNX is viable! {speedup:.1f}x speedup frees {(tf_extrap - onnx_extrap) / 60:.0f} min for CNN blend"
    )
    print(f"   Remaining budget: {remaining:.0f} min (need ~10 min for 1-fold CNN)")
else:
    print(f"❌ ONNX speedup insufficient ({speedup:.1f}x < 3x threshold)")